# 02 - LLaMEA Evolutionary Synthesis Pipeline

Multi-process parallel algorithm discovery across the BBOB problem benchmark matrix with SQLite persistence and auto-resumption.


In [ ]:
import os
import sys
import sqlite3
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd
import numpy as np

# Ensure project root & src are in path
cwd = Path(".").resolve()
root_dir = cwd.parent if cwd.name == "notebooks" else cwd
src_dir = root_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from shared.config import CONFIGS_DIR, DATA_DIR, RESULTS_DIR
from shared.database import initialize_sqlite_storage
from evolution.infra.storage.synthesis_config.repository import SynthesisConfigRepository
from evolution.infra.logging import SynthesisLogger
from evolution.application.synthesis_service import LLaMEASynthesisService
from evolution.infra.llm.client import LLMClient
from evolution.domain.enums import PromptStrategy

# Sanitize legacy database entries if any exist on disk
db_file = DATA_DIR / "db.sqlite3"
if db_file.exists():
    with sqlite3.connect(db_file) as conn:
        conn.execute("UPDATE experiments SET noise_model = 'heteroscedastic' WHERE noise_model = 'multiplicative';")
        conn.commit()

print("[OK] LLaMEA Evolution Pipeline initialized.")

## 1. LLM Provider Connection
Initialize and verify the LLM connection configured in your `.env` file.


In [ ]:
env_path = root_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)

llm_provider = os.getenv("LLM_PROVIDER", "local")

# Direct initialization with live connection validation - halts execution if server is unreachable
llm = LLMClient(llm_provider)
print("=" * 65)
print(f"Initialized Provider: {llm_provider}")
print(f"Model Target:         {getattr(llm, 'model', 'unknown')}")
print("=" * 65)

## 2. Load Configuration & Experiment Matrix (`configs/synthesis.toml`)
Reads the target problems, dimensions, prompt strategies, noise levels, and execution knobs from the unified `synthesis.toml` and audits current completion status in SQLite.

In [ ]:
# 1. Initialize Repositories, Logger, and Evolution Application Service
sqlite_repo = initialize_sqlite_storage()
config_repo = SynthesisConfigRepository()
logger = SynthesisLogger(verbose=True)
synthesis_service = LLaMEASynthesisService(
    sqlite_repo=sqlite_repo,
    config_repo=config_repo,
    llm_client=llm,
    logger=logger,
)

# 2. Audit Matrix Status against SQLite Database for active LLM model
matrix_df, summary = synthesis_service.audit_matrix()
display(matrix_df) if "display" in globals() else print(matrix_df.to_string(index=False))

## 3. Parallel Multi-Process Experiment Execution (With Auto-Resumption & DB Sync)
Automatically queries `data/db.sqlite3` to check which experiments are still pending or incomplete:
1. **Interrupted Runs (`status == 'running'`)**: Fetched from the DB and resumed directly from their last completed iteration without needing re-configuration.
2. **Completed Runs (`status == 'completed'`)**: Automatically skipped if they already produced a valid champion algorithm.
3. **Failed Synthesis Runs (`best_final_error is null`)**: Automatically retried if `retry_failed_synthesis = true`.
4. **Pending Runs**: Created as fresh tasks for the remaining required runs.

All tasks execute concurrently across worker processes using `TaskOrchestrator` (`ProcessPoolExecutor`) with safe SQLite WAL concurrency.

In [ ]:
# Execute evolutionary synthesis tasks in parallel across worker processes
results = synthesis_service.run_synthesis()